In [ ]:
CAMERA_NAME = 'TAPO'
DATE_OVERRIDE = None

In [ ]:
# ── Environment Detection (Colab vs GitHub Actions) ──────────────
import os, re as _re

RUNNING_IN_CI = os.getenv('GITHUB_ACTIONS') == 'true'
CAMERA_NAME = os.environ.get('CAMERA_NAME', CAMERA_NAME).upper()
DATE_OVERRIDE = os.environ.get('DATE_OVERRIDE', DATE_OVERRIDE)
if DATE_OVERRIDE:
    # Force to string and remove any decimal points if passed as float
    DATE_OVERRIDE = str(DATE_OVERRIDE).split('.')[0].strip()

DRIVE_CSV_DATA = []  # Global data for cross-camera checks

# Feeding window: only clips whose filename timestamp falls in this range are processed.
# Filenames are motion_YYYYMMDD_HHMMSS_Xm_Ys.mp4 using Pi local time (CET/CEST).
FEEDING_WINDOW_START = (6, 18)   # (hour, minute)
FEEDING_WINDOW_END   = (6, 30)

import pytz
from datetime import datetime

def _in_feeding_window(filename, start, end):
    m = _re.match(r'motion_(\d{8})_(\d{2})(\d{2})\d{2}', filename)
    if not m:
        print(f"  [SKIP] {filename}: Regex mismatch")
        return False

    cet_tz = pytz.timezone('Europe/Amsterdam')
    # ENSURE today_str IS ALWAYS STRING
    today_str = str(DATE_OVERRIDE) if DATE_OVERRIDE else datetime.now(cet_tz).strftime('%Y%m%d')
    file_date = m.group(1)

    if file_date != today_str:
        print(f"  [SKIP] {filename}: Date mismatch ({file_date} != {today_str})")
        return False

    file_min  = int(m.group(2)) * 60 + int(m.group(3))
    start_min = start[0] * 60 + start[1]
    end_min   = end[0]   * 60 + end[1]
    
    in_window = start_min <= file_min <= end_min
    if not in_window:
        print(f"  [SKIP] {filename}: Time outside window ({m.group(2)}:{m.group(3)})")
        
    return in_window

if RUNNING_IN_CI:
    from google.oauth2 import service_account
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import json as _json, pathlib, tempfile

    _key = _json.loads(os.environ['GDRIVE_SERVICE_ACCOUNT_KEY'])
    _creds = service_account.Credentials.from_service_account_info(
        _key, scopes=['https://www.googleapis.com/auth/drive.readonly']
    )
    _drive = build('drive', 'v3', credentials=_creds)

    GDRIVE_UPLOAD_FOLDER_ID = os.environ.get('GDRIVE_UPLOAD_FOLDER_ID', '').strip()
    if not GDRIVE_UPLOAD_FOLDER_ID:
        GDRIVE_UPLOAD_FOLDER_ID = 'root'
    CI_DOWNLOAD_DIR = pathlib.Path(tempfile.mkdtemp()) / 'videos'
    CI_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    _tz = pytz.timezone('Europe/Amsterdam')
    _search_date = DATE_OVERRIDE if DATE_OVERRIDE else datetime.now(_tz).strftime('%Y%m%d')
    print(f'[{CAMERA_NAME}] Querying Drive for videos in folder: {GDRIVE_UPLOAD_FOLDER_ID}')
    print(f'[{CAMERA_NAME}] Date Filter: {_search_date}')
    _q = f"'{GDRIVE_UPLOAD_FOLDER_ID}' in parents and mimeType='video/mp4' and name contains '{_search_date}' and trashed=false"
    print(f"DEBUG: GDRIVE_UPLOAD_FOLDER_ID length: {len(GDRIVE_UPLOAD_FOLDER_ID)}")
    print(f"DEBUG: FIRST 4 CHARS: {GDRIVE_UPLOAD_FOLDER_ID[:4]}")
    _results = _drive.files().list(
        pageSize=1000,
        q=_q,
        fields='files(id, name, modifiedTime)',
        orderBy='name',
    ).execute()
    _all_files = _results.get('files', [])

    if not _all_files:
        print(f'[{CAMERA_NAME}] ⚠️ No mp4 files found in folder {GDRIVE_UPLOAD_FOLDER_ID} for date {_search_date}.')
        # Optional: try a broader search to see if anything exists
        _test_q = f"'{GDRIVE_UPLOAD_FOLDER_ID}' in parents and trashed=false"
        _test_res = _drive.files().list(pageSize=10, q=_test_q, fields='files(name)').execute()
        _test_files = [f['name'] for f in _test_res.get('files', [])]
        print(f'[{CAMERA_NAME}] Sample of files found in that folder (any type): {_test_files}')
    _feeding_files = [
        f for f in _all_files
        if _in_feeding_window(f['name'], FEEDING_WINDOW_START, FEEDING_WINDOW_END)
    ]
    print(f"[{CAMERA_NAME}] Found {len(_all_files)} total clips, {len(_feeding_files)} in feeding window "
          f"(Date: {_search_date} | {FEEDING_WINDOW_START[0]:02d}:{FEEDING_WINDOW_START[1]:02d}–"
          f"{FEEDING_WINDOW_END[0]:02d}:{FEEDING_WINDOW_END[1]:02d})")

    downloaded_paths = []
    video_paths = []
    if not _feeding_files:
        print(f'[{CAMERA_NAME}] ℹ️ No clips found in feeding window for {_search_date}. Nothing to process.')
    else:
        for _f in _feeding_files:
            _dest = CI_DOWNLOAD_DIR / _f['name']
            if not _dest.exists():
                _req = _drive.files().get_media(fileId=_f['id'])
                with open(_dest, 'wb') as _fh:
                    _dl = MediaIoBaseDownload(_fh, _req)
                    _done = False
                    while not _done:
                        _, _done = _dl.next_chunk()
            print(f'  Downloaded: {_f["name"]}')
            downloaded_paths.append(str(_dest))
        video_paths = downloaded_paths

    SOURCE_DIR = str(CI_DOWNLOAD_DIR)
    print(f'CI mode: SOURCE_DIR = {SOURCE_DIR}')

    # -- Cross-camera check: download feeding_log.csv --
    DRIVE_CSV_DATA = []
    GDRIVE_OUTPUT_FOLDER_ID = os.environ.get('GDRIVE_OUTPUT_FOLDER_ID', '').strip()
    if GDRIVE_OUTPUT_FOLDER_ID:
        _existing_csv = _drive.files().list(
            q=f"'{GDRIVE_OUTPUT_FOLDER_ID}' in parents and name='feeding_log.csv' and trashed=false",
            fields='files(id)'
        ).execute().get('files', [])
        if _existing_csv:
            import io, csv
            _req = _drive.files().get_media(fileId=_existing_csv[0]['id'])
            _content = _req.execute()
            _decoded = _content.decode('utf-8')
            DRIVE_CSV_DATA = list(csv.DictReader(io.StringIO(_decoded)))
            print(f"[{CAMERA_NAME}] Loaded {len(DRIVE_CSV_DATA)} rows from feeding_log.csv for cross-camera check.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    if not RUNNING_IN_CI:
        SOURCE_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/TAPO_autoupload'
    print(f'Colab mode: SOURCE_DIR = {SOURCE_DIR}')


In [ ]:
# ── Stitch feeding-window clips into one video (CI only) ──────────
import os as _os2, re as _re2, subprocess as _sp
from pathlib import Path as _Path2
from datetime import datetime as _dt2, timedelta as _td2

STITCH_GAP_SECONDS = 10

def _parse_clip_times(filename):
    m = _re2.match(r'motion_(\d{8})_(\d{6})(?:_(\d+)m)?_(\d+)s', _Path2(filename).stem)
    if not m:
        return None, None
    start = _dt2.strptime(f"{m.group(1)}{m.group(2)}", "%Y%m%d%H%M%S")
    minutes = int(m.group(3)) if m.group(3) else 0
    seconds = int(m.group(4))
    return start, start + _td2(minutes=minutes, seconds=seconds)

def _group_by_gap(paths, gap_sec=STITCH_GAP_SECONDS):
    if not paths:
        return []
    paths = sorted(paths, key=lambda p: p.name)
    timed = [(p, *_parse_clip_times(p.name)) for p in paths]
    groups, cur = [], [timed[0]]
    for i in range(1, len(timed)):
        prev_end = cur[-1][2]
        curr_start = timed[i][1]
        if prev_end and curr_start:
            gap = (curr_start - prev_end).total_seconds()
        else:
            gap = gap_sec + 1
        if gap <= gap_sec:
            cur.append(timed[i])
        else:
            groups.append([t[0] for t in cur])
            cur = [timed[i]]
    groups.append([t[0] for t in cur])
    return groups

if RUNNING_IN_CI:
    merged_sources = {}  # merged_filename -> [source clip names]
    video_paths = []

    groups = _group_by_gap([_Path2(p) for p in downloaded_paths])
    print(f"ℹ️ {len(downloaded_paths)} clip(s) grouped into {len(groups)} feeding event(s) (gap threshold: {STITCH_GAP_SECONDS}s)")

    for _gi, _group in enumerate(groups):
        if len(_group) > 1:
            _list_file = _os2.path.join(SOURCE_DIR, f'_concat_{_gi}.txt')
            with open(_list_file, 'w') as _lf:
                _lf.write('\n'.join(f"file '{p}'" for p in _group))
            _mname = f"feeding_merged_{_gi}.mp4" if len(groups) > 1 else "feeding_merged.mp4"
            _merged = _os2.path.join(SOURCE_DIR, _mname)
            _r = _sp.run(
                ['ffmpeg', '-f', 'concat', '-safe', '0', '-i', _list_file,
                 '-c', 'copy', '-y', _merged],
                capture_output=True, text=True
            )
            if _r.returncode == 0:
                video_paths.append(_Path2(_merged))
                merged_sources[_mname] = [p.name for p in _group]
                _os2.remove(_list_file)
                print(f"  ✅ Event {_gi+1}: merged {len(_group)} clips → {_mname}")
            else:
                print(f"  ⚠️ Merge failed for event {_gi+1}:\n{_r.stderr}\n  Falling back to individual clips")
                video_paths.extend(_group)
        else:
            video_paths.append(_group[0])
            print(f"  ℹ️ Event {_gi+1}: single clip {_group[0].name}")

<a href="https://colab.research.google.com/github/iknoest/fair-feeder/blob/claude%2Fcat-feeder-dataset-v13-yQuRz/smoketest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fair Feeder V13 — Smoke Test & Feeding Analysis

Run YOLOv11 inference on test images and videos from the Tapo IR camera.

**What this notebook does:**
1. Validates model accuracy on the test split (mAP, per-class AP50)
2. Benchmarks inference speed
3. Runs inference on images/videos from Google Drive
4. Tracks feeding behaviour: kibble count, cat-at-bowl, Dan's hand episodes
5. Extracts timestamps from Tapo's burned-in OSD via OCR
6. Saves text summary + snapshots to Google Drive
7. Sends results (summary, snapshots, video) to Telegram

**Secret management:** API keys are stored in [Infisical](https://app.infisical.com).
Infisical credentials (`INFISICAL_ID`, `INFISICAL_SECRET`, `INFISICAL_PROJECT_ID`) are stored in Colab Secrets.

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics roboflow easyocr tqdm requests infisicalsdk
!nvidia-smi

In [ ]:
# ── Load secrets from Infisical (Colab) or env vars (CI) ───────────
if not RUNNING_IN_CI:
    from google.colab import userdata
    from infisical_sdk import InfisicalSDKClient

    _client = InfisicalSDKClient(host="https://app.infisical.com")
    _client.auth.universal_auth.login(
        client_id=userdata.get('INFISICAL_ID'),
        client_secret=userdata.get('INFISICAL_SECRET'),
    )
    _proj = userdata.get('INFISICAL_PROJECT_ID')

    def _get_secret(name):
        return _client.secrets.get_secret_by_name(
            secret_name=name,
            project_id=_proj,
            environment_slug="dev",
            secret_path="/",
        ).secretValue

    ROBOFLOW_API_KEY = _get_secret("Roboflow")
    BOT_TOKEN        = _get_secret("TelegramBotToken")
    MY_CHAT_ID       = _get_secret("TelegramChatId")
    print("✅ Secrets loaded from Infisical")
    print(f"   Roboflow key : {ROBOFLOW_API_KEY[:5]}...")
    print(f"   Telegram bot : {BOT_TOKEN[:10]}...")
else:
    # CI: secrets injected as environment variables via GitHub Actions
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
    BOT_TOKEN  = os.environ['TelegramBotToken']
    MY_CHAT_ID = os.environ['TelegramChatId']
    print("✅ Secrets loaded from environment (CI mode)")
    print(f"   Telegram bot : {BOT_TOKEN[:10]}...")


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from IPython.display import display, Video, Image as IPImage
from collections import defaultdict
from PIL import Image
import os, glob, time, re, json, requests
from tqdm.auto import tqdm
import easyocr

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
if RUNNING_IN_CI:
    import tempfile as _tempfile
    _ci_root = pathlib.Path(_tempfile.mkdtemp())
    # Download YOLO model from Drive using service account
    _model_file_id = os.environ.get('GDRIVE_MODEL_FILE_ID', '')
    MODEL_PATH = str(_ci_root / 'fair_feeder_v14_yolov11s.pt')
    if _model_file_id:
        from googleapiclient.http import MediaIoBaseDownload as _MIBD
        _req = _drive.files().get_media(fileId=_model_file_id)
        with open(MODEL_PATH, 'wb') as _fh:
            _dl = _MIBD(_fh, _req)
            _done = False
            while not _done:
                _, _done = _dl.next_chunk()
        print(f'  Downloaded model: {MODEL_PATH}')
    else:
        print('⚠️  GDRIVE_MODEL_FILE_ID not set — set this secret to enable analysis')
    OUTPUT_DIR = str(_ci_root / 'output')
else:
    MODEL_PATH = '/content/drive/MyDrive/Fun Project/Cat monitor/model/fair_feeder_v14_yolov11s.pt'
    SOURCE_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel'
    OUTPUT_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Class colours (RGB for display, BGR for OpenCV) ───────────
CLASS_COLORS_RGB = {
    "Dan":      (0,   255, 206),
    "Bowl":     (134,  34, 255),
    "Dan_hand": (0,   16,  255),
    "Kibble":   (252, 244,   0),
    "Sanbo":    (255, 128,   0),
}
CLASS_ORDER = ["Dan", "Sanbo", "Dan_hand", "Bowl", "Kibble"]

CLASS_ROW_BG = {
    "Dan":      "#003D2F",
    "Sanbo":    "#3D2000",
    "Dan_hand": "#00073D",
    "Kibble":   "#3D3A00",
    "Bowl":     "#250050",
}

CONF_THRESHOLD      = 0.45
IOU_THRESHOLD       = 0.20
IMGSZ               = 1280
GRAY_DIFF_THRESHOLD = 15
OVERLAP_IOU_THRESHOLD = 0.10
DAN_HAND_CONF_THRESHOLD = 0.50
DAN_HAND_GAP_SECONDS    = 2.0
DAN_HAND_MIN_SECONDS    = 0.5
EATING_KIBBLE_DROP = 1
FRAME_SKIP = 7
EMPTY_BOWL_EXIT_SECONDS = 9999
KIBBLE_SNAPSHOT_STABLE_FRAMES = 3
FOOD_FINISHED_KIBBLE_MAX = 1
TIMESTAMP_CROP = (0, 0, 800, 80)
OCR_EVERY_N_FRAMES = 10
USE_CACHE    = True
FORCE_RERUN  = False
import pickle
SANBO_MIN_CONSECUTIVE_FRAMES = 5
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

print("✅ Config loaded.")
print(f"   Model  : {MODEL_PATH}")
print(f"   Source : {SOURCE_DIR}")
print(f"   Output : {OUTPUT_DIR}")


In [ ]:
model = YOLO(MODEL_PATH)

# FIX: Use model.names for correct class→index mapping
# model.names = {0: 'Bowl', 1: 'Dan', 2: 'Dan_hand', 3: 'Kibble', 4: 'Sanbo'}
CLASS_NAMES = model.names
print("✅ Model loaded.")
print(f"   Classes: {CLASS_NAMES}")

In [ ]:
# ───────────────────────────────────────────────────────────────
# OCR + Helper Functions
# ───────────────────────────────────────────────────────────────

# --- OCR reader for Tapo timestamp ---
ocr_reader = easyocr.Reader(['en'], gpu=False)
TIMESTAMP_REGEX = re.compile(r'(\d{4}[-/]\d{2}[-/]\d{2})\s*(\d{1,2}:\d{2}:\d{1,2})')


def _calculate_timestamp(filename, frame_idx, fps):
    """Derive timestamp from filename like motion_20260518_213615.mp4"""
    if not filename:
        return ""
    # Support motion_YYYYMMDD_HHMMSS or just YYYYMMDD_HHMMSS
    m = re.search(r'(\d{8})_(\d{6})', filename)
    if not m:
        # Try finding just a date-like pattern as a last resort
        m_date = re.search(r'(\d{8})', filename)
        if m_date:
            return f"{m_date.group(1)[:4]}-{m_date.group(1)[4:6]}-{m_date.group(1)[6:8]} 06:00:00"
        return ""
    
    from datetime import datetime, timedelta
    try:
        start_dt = datetime.strptime(f"{m.group(1)}{m.group(2)}", "%Y%m%d%H%M%S")
        current_dt = start_dt + timedelta(seconds=frame_idx / fps)
        return current_dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception as e:
        print(f"  [DEBUG] Timestamp calc error for {filename}: {e}")
        return ""


def extract_timestamp(frame, filename=None, frame_idx=None, fps=None):
    """OCR the Tapo burned-in timestamp, or fall back to calculation for Logitech."""
    if CAMERA_NAME == 'LOGITECH':
        return _calculate_timestamp(filename, frame_idx, fps)

    # TAPO: Try OCR first
    x1, y1, x2, y2 = TIMESTAMP_CROP
    crop = frame[y1:y2, x1:x2]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
    results = ocr_reader.readtext(thresh, detail=0)
    # Strip all spaces ? easyOCR often reads chars individually: "2 0 2 6 - 0 1"
    text = "".join(results).replace(" ", "").replace("|:", ":1").replace("|", "1")
    m = TIMESTAMP_REGEX.search(text)
    if m:
        date_part = m.group(1).replace("/", "-")
        time_part = m.group(2)
        # Zero-pad partial components: H:MM:S ? HH:MM:SS
        h, mi, s = time_part.split(":")
        time_part = f"{h.zfill(2)}:{mi}:{s.zfill(2)}"
        return f"{date_part} {time_part}"
    
    # Fallback to calculation if OCR fails
    if filename is not None and frame_idx is not None and fps is not None:
        return _calculate_timestamp(filename, frame_idx, fps)
    return ""


def classify_file(filepath):
    """Return 'image', 'video', or None based on file extension."""
    ext = Path(filepath).suffix.lower()
    if ext in IMAGE_EXTENSIONS:
        return "image"
    elif ext in VIDEO_EXTENSIONS:
        return "video"
    return None


def detect_image_type(img_bgr):
    """
    Returns 'color' or 'ir'.
    Tapo IR images are grayscale stored as BGR where R≈G≈B.
    """
    b, g, r = cv2.split(img_bgr)
    b, g, r = b.astype(int), g.astype(int), r.astype(int)
    max_diff = max(
        np.mean(np.abs(r - g)),
        np.mean(np.abs(r - b)),
        np.mean(np.abs(g - b)),
    )
    return "ir" if max_diff < GRAY_DIFF_THRESHOLD else "color"


def prepare_for_inference(img_bgr, image_type):
    """Color image → gray then back to 3-ch BGR for YOLO. IR → pass through."""
    if image_type == "color":
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    return img_bgr


def get_color_bgr(class_name):
    r, g, b = CLASS_COLORS_RGB.get(class_name, (255, 255, 255))
    return (b, g, r)


def bbox_iou(box_a, box_b):
    """Compute IoU between two dicts with x1,y1,x2,y2 keys."""
    xa1 = max(box_a["x1"], box_b["x1"])
    ya1 = max(box_a["y1"], box_b["y1"])
    xa2 = min(box_a["x2"], box_b["x2"])
    ya2 = min(box_a["y2"], box_b["y2"])
    inter = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_a = (box_a["x2"] - box_a["x1"]) * (box_a["y2"] - box_a["y1"])
    area_b = (box_b["x2"] - box_b["x1"]) * (box_b["y2"] - box_b["y1"])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def parse_results(results):
    """Parse YOLO results into a list of detection dicts."""
    detections = []
    for result in results:
        if result.boxes is None:
            continue
        for box in result.boxes:
            cls_id = int(box.cls.item())
            detections.append({
                "class_name": model.names[cls_id],
                "conf": float(box.conf.item()),
                "x1": int(box.xyxy[0][0].item()),
                "y1": int(box.xyxy[0][1].item()),
                "x2": int(box.xyxy[0][2].item()),
                "y2": int(box.xyxy[0][3].item()),
            })
    return detections


def draw_boxes(img_bgr, detections, show_label=False):
    """Draw boxes on a copy of img_bgr."""
    out = img_bgr.copy()
    thick = max(2, int(img_bgr.shape[1] / 600))
    for det in detections:
        name = det["class_name"]
        conf = det["conf"]
        x1, y1, x2, y2 = det["x1"], det["y1"], det["x2"], det["y2"]
        bgr = get_color_bgr(name)
        cv2.rectangle(out, (x1, y1), (x2, y2), bgr, thick)
        if show_label:
            label = f"{name} {conf*100:.0f}%"
            font = cv2.FONT_HERSHEY_SIMPLEX
            fscale = max(0.5, img_bgr.shape[1] / 2000)
            fthick = max(1, thick - 1)
            (tw, th), _ = cv2.getTextSize(label, font, fscale, fthick)
            pad = 4
            cv2.rectangle(out, (x1, max(0, y1 - th - pad*2)), (x1 + tw + pad, y1), bgr, -1)
            cv2.putText(out, label, (x1 + pad//2, y1 - pad), font, fscale, (0,0,0), fthick, cv2.LINE_AA)
    return out


def compute_stats(values):
    if not values:
        return {"count": 0, "min": None, "max": None, "median": None, "variance": None}
    arr = np.array(values, dtype=float)
    return {
        "count": len(arr),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "median": float(np.median(arr)),
        "variance": float(np.var(arr)),
    }


def print_image_summary(img_name, image_type, stats_by_class):
    tag = "🌈 COLOR (converted to gray for inference)" if image_type == "color" else "🌙 IR"
    print(f"\n{'─'*65}")
    print(f"  📷  {img_name}  [{tag}]")
    print(f"{'─'*65}")
    print(f"  {'Class':<12} {'Count':>6}  {'Min%':>6}  {'Max%':>6}  {'Med%':>6}  {'Variance':>9}")
    print(f"  {'─'*59}")
    total = 0
    for name in CLASS_ORDER:
        s = stats_by_class.get(name, compute_stats([]))
        if s["count"] > 0:
            print(f"  {name:<12} {s['count']:>6}  "
                  f"{s['min']*100:>5.1f}%  "
                  f"{s['max']*100:>5.1f}%  "
                  f"{s['median']*100:>5.1f}%  "
                  f"{s['variance']*10000:>8.2f}")
            total += s["count"]
        else:
            print(f"  {name:<12} {'—':>6}")
    print(f"  {'─'*59}")
    print(f"  {'TOTAL':<12} {total:>6}")


def show_dual_output(img_name, image_type, img_plain, img_labeled, stats_by_class):
    left = cv2.cvtColor(img_plain, cv2.COLOR_BGR2RGB)
    right = cv2.cvtColor(img_labeled, cv2.COLOR_BGR2RGB)
    tag = "COLOR → gray inference" if image_type == "color" else "IR"
    fig, axes = plt.subplots(1, 2, figsize=(22, 7), facecolor="#1a1a1a")
    for ax, img, title in zip(axes, [left, right], ["Boxes only", "Boxes + Confidence"]):
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=12, pad=8)
        ax.axis("off")
    fig.suptitle(f"{img_name}  [{tag}]", color="white", fontsize=13, y=1.01, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"preview_{Path(img_name).stem}.jpg"),
                dpi=120, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print_image_summary(img_name, image_type, stats_by_class)


def render_final_table(total_images, color_count, ir_count, all_stats):
    """Styled summary table saved as SUMMARY_TABLE.jpg."""
    col_labels = ["Class", "Detections", "Min %", "Max %", "Median %", "Variance"]
    col_widths = [0.18, 0.14, 0.12, 0.12, 0.14, 0.14]
    col_align = ["left", "center", "center", "center", "center", "center"]
    rows = []
    row_meta = []
    grand_all = []
    for name in CLASS_ORDER:
        confs = all_stats.get(name, [])
        grand_all.extend(confs)
        s = compute_stats(confs)
        rows.append([
            name,
            str(s["count"]),
            f"{s['min']*100:.1f}%" if s["count"] else "—",
            f"{s['max']*100:.1f}%" if s["count"] else "—",
            f"{s['median']*100:.1f}%" if s["count"] else "—",
            f"{s['variance']*10000:.2f}" if s["count"] else "—",
        ])
        row_meta.append((name, CLASS_ROW_BG.get(name, "#1E1E1E")))
    gs = compute_stats(grand_all)
    rows.append([
        "ALL CLASSES",
        str(gs["count"]),
        f"{gs['min']*100:.1f}%" if gs["count"] else "—",
        f"{gs['max']*100:.1f}%" if gs["count"] else "—",
        f"{gs['median']*100:.1f}%" if gs["count"] else "—",
        f"{gs['variance']*10000:.2f}" if gs["count"] else "—",
    ])
    row_meta.append(("__total__", "#2C2C2C"))
    n_data_rows = len(rows)
    row_h = 0.11
    header_h = 0.13
    top_pad = 0.25
    bottom_pad = 0.12
    total_h = top_pad + header_h + n_data_rows * row_h + bottom_pad
    fig, ax = plt.subplots(figsize=(13, total_h * 6), facecolor="#1a1a1a")
    ax.set_facecolor("#1a1a1a")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, total_h)
    ax.axis("off")
    ax.text(0.5, total_h - 0.05, "Fair Feeder V13 — Smoke Test Summary",
            ha="center", va="top", color="white", fontsize=14, fontweight="bold")
    ax.text(0.5, total_h - 0.13,
            f"Images analysed: {total_images}     🌈 Color: {color_count}     🌙 IR: {ir_count}",
            ha="center", va="top", color="#AAAAAA", fontsize=10)
    x_starts = []
    x = 0.01
    for w in col_widths:
        x_starts.append(x)
        x += w
    header_y = total_h - top_pad
    header_rect = patches.FancyBboxPatch(
        (0.01, header_y - header_h + 0.02), 0.97, header_h,
        boxstyle="round,pad=0.005", linewidth=0, facecolor="#1F4E78")
    ax.add_patch(header_rect)
    for xi, w, label, align in zip(x_starts, col_widths, col_labels, col_align):
        tx = xi + w/2 if align == "center" else xi + 0.01
        ax.text(tx, header_y - header_h/2 + 0.02, label,
                ha=align if align != "left" else "left", va="center",
                color="white", fontsize=10, fontweight="bold")
    for i, (row, (rname, bg)) in enumerate(zip(rows, row_meta)):
        y_top = header_y - header_h - i * row_h
        is_total = (rname == "__total__")
        rect = patches.FancyBboxPatch(
            (0.01, y_top - row_h + 0.005), 0.97, row_h - 0.01,
            boxstyle="round,pad=0.003", linewidth=0, facecolor=bg)
        ax.add_patch(rect)
        for j, (xi, w, val, align) in enumerate(zip(x_starts, col_widths, row, col_align)):
            tx = xi + w/2 if align == "center" else xi + 0.01
            if j == 0:
                r_c, g_c, b_c = CLASS_COLORS_RGB.get(rname, (255, 215, 0)) if not is_total else (255, 215, 0)
                txt_color = "#%02X%02X%02X" % (r_c, g_c, b_c) if not is_total else "#FFD700"
            else:
                txt_color = "#FFD700" if is_total else "white"
            ax.text(tx, y_top - row_h/2 + 0.005, val,
                    ha="left" if align == "left" else "center", va="center",
                    color=txt_color, fontsize=10,
                    fontweight="bold" if is_total else "normal")
    ax.text(0.5, 0.04,
            f"conf_threshold={CONF_THRESHOLD}  |  iou_threshold={IOU_THRESHOLD}  |  imgsz={IMGSZ}  |  Variance × 10⁴",
            ha="center", va="bottom", color="#555555", fontsize=8)
    table_path = os.path.join(OUTPUT_DIR, "SUMMARY_TABLE.jpg")
    plt.savefig(table_path, dpi=150, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print(f"\n✅  Summary table saved → {table_path}")


print("✅ Helper functions defined.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Feeding Behaviour Tracker
# ───────────────────────────────────────────────────────────────

class FeedingTracker:
    """Tracks feeding events across video frames.

    Eating logic — phase-based attribution:
      1. Merge overlapping cat-at-bowl episodes into "feeding phases"
      2. Measure kibble only at clear frames (no cat/hand occluding bowl)
      3. Attribute kibble drops to cat(s) present, proportional to
         their bowl-overlap time within each phase
    """

    def __init__(self, fps=25.0, sanbo_min_frames=1, episode_offset=0):
        self.fps = fps
        self.dan_hand_gap_frames = int(DAN_HAND_GAP_SECONDS * fps)
        self.dan_hand_min_frames = int(DAN_HAND_MIN_SECONDS * fps)
        self.sanbo_min_frames = sanbo_min_frames
        self.episode_offset = episode_offset

        # Per-frame arrays
        self.kibble_counts = []       # int per frame
        self.dan_at_bowl = []         # bool per frame
        self.sanbo_at_bowl = []       # bool per frame
        self.dan_hand_present = []    # bool per frame (filtered: bowl overlap + conf + Dan body)
        self.dan_hand_confs = []      # float per frame (max conf, 0 if no valid hand)
        self.timestamps = []          # OCR string per frame

        # Events
        self.dan_first_frame = None
        self.dan_first_timestamp = None
        self.sanbo_first_frame = None
        self.sanbo_first_timestamp = None
        self.snapshots = {}  # {"sanbo_arrival": img, "dan_hand_ep0": img, ...}

        # Internal: track best-confidence frame per Dan_hand episode
        self._current_hand_ep_idx = -1
        self._current_hand_best_conf = 0.0

        # Kibble-dispensed snapshot: keep clean pre-cat frames, then watch after hand episode
        self._pre_cat_kibble_buf = []       # consecutive no-cat frames with visible kibble
        self._pre_cat_kibble_snapshot = None
        self._pre_cat_kibble_best = None
        self._kibble_watch_active = False
        self._kibble_watch_ep_idx = 0
        self._kibble_watch_start_frame = 0
        self._kibble_watch_stable_buf = []  # all watched frames for fallback
        self._kibble_watch_clear_buf = []   # consecutive clear frames for stable snapshot

    def process_frame(self, frame_idx, detections, timestamp_str, frame_img):
        """Process one frame's detections."""
        # Kibble count
        kibble_count = sum(1 for d in detections if d["class_name"] == "Kibble")
        self.kibble_counts.append(kibble_count)
        self.timestamps.append(timestamp_str)

        # Find Bowl bbox (use largest if multiple)
        bowl_boxes = [d for d in detections if d["class_name"] == "Bowl"]
        bowl_box = max(bowl_boxes, key=lambda d: (d["x2"]-d["x1"])*(d["y2"]-d["y1"])) if bowl_boxes else None

        # Cat-at-bowl overlap (IoU > 10%)
        dan_at = False
        sanbo_at = False
        if bowl_box:
            for d in detections:
                if d["class_name"] == "Dan" and bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                    dan_at = True
                elif d["class_name"] == "Sanbo" and bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                    sanbo_at = True
        self.dan_at_bowl.append(dan_at)
        self.sanbo_at_bowl.append(sanbo_at)

        # Dan body presence check (used for Dan_hand co-detection + first appearance)
        dan_here = any(d["class_name"] == "Dan" for d in detections)

        # Dan_hand: require conf + bowl overlap + Dan body in same frame
        dan_hand = False
        dan_hand_conf = 0.0
        if bowl_box and dan_here:
            for d in detections:
                if d["class_name"] == "Dan_hand" and d["conf"] >= DAN_HAND_CONF_THRESHOLD:
                    if bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                        dan_hand = True
                        dan_hand_conf = max(dan_hand_conf, d["conf"])
        self.dan_hand_present.append(dan_hand)
        self.dan_hand_confs.append(dan_hand_conf)

        stable_required = int(globals().get("KIBBLE_SNAPSHOT_STABLE_FRAMES", 3))
        prev_hand_for_clean = self.dan_hand_present[-2] if len(self.dan_hand_present) >= 2 else False
        if (kibble_count > 0 and not dan_at and not sanbo_at
                and not dan_hand and not prev_hand_for_clean):
            clean_item = (frame_idx, kibble_count, frame_img.copy(), detections)
            self._pre_cat_kibble_buf.append(clean_item)
            self._pre_cat_kibble_buf = self._pre_cat_kibble_buf[-stable_required:]
            if (self._pre_cat_kibble_best is None
                    or kibble_count > self._pre_cat_kibble_best[1]):
                self._pre_cat_kibble_best = clean_item
            recent_clean = self._pre_cat_kibble_buf[-stable_required:]
            if len(recent_clean) >= stable_required:
                clean_counts = [item[1] for item in recent_clean]
                if max(clean_counts) == min(clean_counts):
                    self._pre_cat_kibble_snapshot = recent_clean[-1]
        else:
            self._pre_cat_kibble_buf = []

        if (dan_at or sanbo_at) and not any(
                k.startswith("kibble_dispensed") for k in self.snapshots):
            pre_cat_candidates = [item for item in (
                self._pre_cat_kibble_snapshot, self._pre_cat_kibble_best) if item is not None]
            if pre_cat_candidates:
                pre_cat = max(pre_cat_candidates, key=lambda x: x[1])
                self.snapshots["kibble_dispensed"] = draw_boxes(
                    pre_cat[2], pre_cat[3], show_label=False)

        # Dan arrival = first frame where Dan overlaps Bowl.
        if dan_at and self.dan_first_frame is None:
            self.dan_first_frame = frame_idx
            self.dan_first_timestamp = timestamp_str

        # Sanbo arrival = consecutive frames where Sanbo overlaps Bowl
        # Prevents false alarms from single-frame Dan/Sanbo body overlaps
        if sanbo_at:
            self._sanbo_consec_count = getattr(self, '_sanbo_consec_count', 0) + 1
        else:
            self._sanbo_consec_count = 0

        if (sanbo_at and self.sanbo_first_frame is None
                and self._sanbo_consec_count >= self.sanbo_min_frames):
            self.sanbo_first_frame = frame_idx
            self.sanbo_first_timestamp = timestamp_str
            annotated_snap = draw_boxes(frame_img, detections, show_label=True)
            self.snapshots["sanbo_arrival"] = annotated_snap

        # Dan_hand snapshot: capture highest-confidence frame per episode
        if dan_hand:
            prev_hand = self.dan_hand_present[-2] if len(self.dan_hand_present) >= 2 else False
            if not prev_hand:
                # New episode starting
                ep_idx = self._count_hand_episodes_so_far()
                self._current_hand_ep_idx = ep_idx
                self._current_hand_best_conf = dan_hand_conf
                annotated_snap = draw_boxes(frame_img, detections, show_label=True)
                self.snapshots[f"dan_hand_ep{ep_idx + self.episode_offset}"] = annotated_snap
            else:
                # Continuing episode: update snapshot if higher confidence
                if dan_hand_conf > self._current_hand_best_conf:
                    self._current_hand_best_conf = dan_hand_conf
                    annotated_snap = draw_boxes(frame_img, detections, show_label=True)
                    self.snapshots[f"dan_hand_ep{self._current_hand_ep_idx + self.episode_offset}"] = annotated_snap

        # ── Kibble-dispensed snapshot logic ────────────────────
        # Detect falling edge of Dan_hand (episode just ended)
        if len(self.dan_hand_present) >= 2 and \
           not self.dan_hand_present[-1] and self.dan_hand_present[-2]:
            self._kibble_watch_active = True
            self._kibble_watch_ep_idx = self._current_hand_ep_idx
            self._kibble_watch_start_frame = frame_idx
            self._kibble_watch_stable_buf = []
            self._kibble_watch_clear_buf = []
        # Kibble-dispensed snapshot: wait for a stable clear kibble count.
        # Fallback: best-kibble frame (any cat state) within 5s if Dan never clears the bowl.
        if self._kibble_watch_active:
            frames_elapsed = frame_idx - self._kibble_watch_start_frame
            timeout_slow = int(self.fps * 5)
            self._kibble_watch_stable_buf.append(
                (frame_idx, kibble_count, frame_img.copy(), detections))

            ep_key = f"kibble_dispensed_ep{self._kibble_watch_ep_idx + self.episode_offset}"
            pre_cat_candidates = [item for item in (
                self._pre_cat_kibble_snapshot, self._pre_cat_kibble_best) if item is not None]
            pre_cat = max(pre_cat_candidates, key=lambda x: x[1]) if pre_cat_candidates else None
            if (dan_at or sanbo_at) and pre_cat is not None and ep_key not in self.snapshots:
                self.snapshots[ep_key] = draw_boxes(
                    pre_cat[2], pre_cat[3], show_label=False)
                self._kibble_watch_active = False
                self._pre_cat_kibble_buf = []
                self._pre_cat_kibble_snapshot = None
                self._pre_cat_kibble_best = None
                return

            if not dan_at and not sanbo_at and not dan_hand:
                self._kibble_watch_clear_buf.append(
                    (frame_idx, kibble_count, frame_img.copy(), detections))
                recent = self._kibble_watch_clear_buf[-stable_required:]
                if len(recent) >= stable_required:
                    counts = [item[1] for item in recent]
                    if min(counts) > 0 and max(counts) == min(counts):
                        self._pre_cat_kibble_snapshot = recent[-1]
                        if (self._pre_cat_kibble_best is None
                                or recent[-1][1] > self._pre_cat_kibble_best[1]):
                            self._pre_cat_kibble_best = recent[-1]
            else:
                self._kibble_watch_clear_buf = []

            # Fallback: 5s elapsed - use highest-kibble frame seen so far
            if self._kibble_watch_active and frames_elapsed >= timeout_slow:
                ep_key = f"kibble_dispensed_ep{self._kibble_watch_ep_idx + self.episode_offset}"
                if ep_key not in self.snapshots:
                    candidates = list(self._kibble_watch_stable_buf)
                    if pre_cat is not None:
                        candidates.append(pre_cat)
                    if candidates:
                        best = max(candidates, key=lambda x: x[1])
                        if best[1] > 0:
                            self.snapshots[ep_key] = draw_boxes(
                                best[2], best[3], show_label=False)
                self._kibble_watch_active = False

    def _count_hand_episodes_so_far(self):
        """Count how many Dan_hand episodes have started so far."""
        episodes = 0
        in_episode = False
        gap = 0
        for present in self.dan_hand_present:
            if present:
                if not in_episode:
                    episodes += 1
                    in_episode = True
                gap = 0
            else:
                gap += 1
                if gap >= self.dan_hand_gap_frames:
                    in_episode = False
        return episodes - 1  # current one is starting, so index = count-1

    def _find_episodes(self, bool_array, gap_frames):
        """Find contiguous True episodes with a minimum gap between them.
        Returns list of (start_frame, end_frame)."""
        episodes = []
        start = None
        gap = 0
        for i, val in enumerate(bool_array):
            if val:
                if start is None:
                    start = i
                gap = 0
            else:
                gap += 1
                if start is not None and gap >= gap_frames:
                    episodes.append((start, i - gap))
                    start = None
        if start is not None:
            episodes.append((start, len(bool_array) - 1))
        return episodes


    def _find_kibble_at_phase_entry(self, phase_start, phase_end, entry_sec=1.0):
        """Kibble count from first N seconds of phase (cat just arrived, minimal eating)."""
        entry_frames = max(1, int(self.fps * entry_sec))
        end_sample = min(phase_start + entry_frames, phase_end + 1)
        counts = self.kibble_counts[phase_start:end_sample]
        return int(np.median(counts)) if counts else None

    def _find_kibble_at_phase_exit(self, phase_start, phase_end, exit_sec=2.0):
        """Kibble count from last N seconds of phase (just before cat left)."""
        exit_frames = max(1, int(self.fps * exit_sec))
        start_sample = max(phase_start, phase_end - exit_frames + 1)
        counts = self.kibble_counts[start_sample:phase_end + 1]
        return int(np.median(counts)) if counts else None

    def _find_clear_kibble_count(self, from_frame, direction="before"):
        """Search for a reliable kibble count from clear frames.

        A 'clear frame' = no cat and no hand bbox overlaps the bowl.
        Searches up to 5 seconds in the given direction. Collects up to
        5 consecutive clear frames for a stable median.

        Returns int or None (if no clear frames found).
        """
        max_search = int(self.fps * 5)
        n = len(self.kibble_counts)

        if direction == "before":
            search_range = range(from_frame - 1, max(from_frame - max_search, -1), -1)
        else:
            search_range = range(from_frame + 1, min(from_frame + max_search, n))

        clear_counts = []
        for i in search_range:
            if 0 <= i < n:
                if not self.dan_at_bowl[i] and not self.sanbo_at_bowl[i] and not self.dan_hand_present[i]:
                    clear_counts.append(self.kibble_counts[i])
                    if len(clear_counts) >= 5:
                        break
                elif clear_counts:
                    # Had clear frames, now hit occluded — stop
                    break

        return int(np.median(clear_counts)) if clear_counts else None

    def _build_feeding_phases(self):
        """Merge all cat-at-bowl episodes into contiguous feeding phases.

        A feeding phase = contiguous period where at least one cat is at
        the bowl. Small gaps (< 1 second) are bridged.

        Returns list of dicts with start, end, dan_frames, sanbo_frames.
        """
        n = len(self.kibble_counts)
        if n == 0:
            return []

        any_at_bowl = [
            self.dan_at_bowl[i] or self.sanbo_at_bowl[i]
            for i in range(n)
        ]

        gap_tolerance = int(self.fps * 1.0)
        raw_phases = self._find_episodes(any_at_bowl, gap_frames=gap_tolerance)

        phases = []
        for start_f, end_f in raw_phases:
            dan_count = sum(1 for i in range(start_f, end_f + 1) if self.dan_at_bowl[i])
            sanbo_count = sum(1 for i in range(start_f, end_f + 1) if self.sanbo_at_bowl[i])
            phases.append({
                'start': start_f,
                'end': end_f,
                'dan_frames': dan_count,
                'sanbo_frames': sanbo_count,
            })

        return phases

    def _valid_timestamp(self, ts):
        return isinstance(ts, str) and TIMESTAMP_REGEX.search(ts) is not None

    def _get_timestamp(self, frame_idx):
        """Get the closest valid timestamp for a frame."""
        if frame_idx < len(self.timestamps) and self._valid_timestamp(self.timestamps[frame_idx]):
            return self.timestamps[frame_idx]
        for offset in range(1, 60):
            for idx in [frame_idx - offset, frame_idx + offset]:
                if 0 <= idx < len(self.timestamps) and self._valid_timestamp(self.timestamps[idx]):
                    return self.timestamps[idx]
        return "??"

    def summarize(self):
        """Compute feeding summary using phase-based eating attribution."""
        n_frames = len(self.kibble_counts)
        if n_frames == 0:
            return {"error": "No frames processed"}

        # ── Smooth kibble counts (rolling median, window=3) ───
        # Reduces flicker from same kibbles being detected/undetected
        if n_frames >= 3:
            smoothed = []
            for i in range(n_frames):
                w_start = max(0, i - 1)
                w_end = min(n_frames, i + 2)
                smoothed.append(int(np.median(self.kibble_counts[w_start:w_end])))
            self.kibble_counts = smoothed

        start_ts = self._get_timestamp(0)
        end_ts = self._get_timestamp(n_frames - 1)

        # ── Dan_hand episodes ─────────────────────────────────
        hand_episodes = self._find_episodes(self.dan_hand_present, self.dan_hand_gap_frames)
        # Filter out episodes shorter than minimum duration
        hand_episodes = [
            (s, e) for s, e in hand_episodes
            if (e - s) >= self.dan_hand_min_frames
        ]

        # Clean up orphaned snapshots for filtered-out episodes
        confirmed_ep_indices = set(i + self.episode_offset for i in range(len(hand_episodes)))
        orphan_keys = [k for k in list(self.snapshots.keys())
                       if k.startswith("dan_hand_ep")
                       and int(k.split("ep")[1]) not in confirmed_ep_indices]
        for k in orphan_keys:
            del self.snapshots[k]
        # Also clean up kibble_dispensed snapshots for orphaned episodes
        orphan_kibble_keys = [k for k in list(self.snapshots.keys())
                              if k.startswith("kibble_dispensed_ep")
                              and int(k.split("ep")[1]) not in confirmed_ep_indices]
        for k in orphan_kibble_keys:
            del self.snapshots[k]

        hand_summary = []
        for start_f, end_f in hand_episodes:
            kb_before = self._find_clear_kibble_count(start_f, direction="before")
            kb_after = self._find_clear_kibble_count(end_f, direction="after")
            kibble_added = 0
            if kb_before is not None and kb_after is not None:
                kibble_added = max(0, kb_after - kb_before)
            hand_summary.append({
                "start_frame": start_f,
                "end_frame": end_f,
                "timestamp": self._get_timestamp(start_f),
                "kibble_added": kibble_added,
            })

        # ── Phase-based eating attribution ────────────────────
        phases = self._build_feeding_phases()

        dan_kibble_eaten = 0
        sanbo_kibble_eaten = 0
        dan_bowl_seconds = 0.0
        sanbo_bowl_seconds = 0.0

        for phase in phases:
            start_f = phase['start']
            end_f = phase['end']
            dan_f = phase['dan_frames']
            sanbo_f = phase['sanbo_frames']

            dan_bowl_seconds += dan_f / self.fps
            sanbo_bowl_seconds += sanbo_f / self.fps

            kb_before = self._find_clear_kibble_count(start_f, direction="before")
            if not kb_before:
                kb_before = self._find_kibble_at_phase_entry(start_f, end_f)
            kb_after = self._find_clear_kibble_count(end_f, direction="after")
            # Fallback: no clear frames after phase (e.g. video ends with cat at bowl)
            if kb_after is None:
                kb_after = self._find_kibble_at_phase_exit(start_f, end_f)

            # Edge case: video starts during feeding, no clear frames before
            if kb_before is None and start_f < int(self.fps * 2):
                kb_before = self.kibble_counts[0]
            # Edge case: video ends during feeding, no clear frames after
            if kb_after is None and end_f > n_frames - int(self.fps * 2):
                kb_after = self.kibble_counts[-1]

            if kb_before is None or kb_after is None:
                continue

            drop = kb_before - kb_after
            if drop < EATING_KIBBLE_DROP:
                continue

            # Attribute to cat(s) present at the bowl during this phase
            if dan_f > 0 and sanbo_f == 0:
                dan_kibble_eaten += drop
            elif sanbo_f > 0 and dan_f == 0:
                sanbo_kibble_eaten += drop
            elif dan_f > 0 and sanbo_f > 0:
                total_f = dan_f + sanbo_f
                dan_kibble_eaten += round(drop * dan_f / total_f)
                sanbo_kibble_eaten += round(drop * sanbo_f / total_f)

        # ── Double-counting guard ─────────────────────────────
        # Best starting estimate = max visible kibble (peak camera angle)
        peak_kibble = max(self.kibble_counts) if self.kibble_counts else None
        last_clear = self._find_clear_kibble_count(n_frames - 1, direction="before")
        if last_clear is None and phases:
            last_clear = self._find_kibble_at_phase_exit(phases[-1]["start"], phases[-1]["end"])
        end_kibble = last_clear
        if peak_kibble is not None and last_clear is not None:
            max_total = max(0, peak_kibble - last_clear)
            total_attr = dan_kibble_eaten + sanbo_kibble_eaten
            if total_attr > max_total and total_attr > 0:
                scale = max_total / total_attr
                dan_kibble_eaten = round(dan_kibble_eaten * scale)
                sanbo_kibble_eaten = round(sanbo_kibble_eaten * scale)

        return {
            "n_frames": n_frames,
            "duration_sec": n_frames / self.fps,
            "start_ts": start_ts,
            "end_ts": end_ts,
            "dan_first_ts": self._get_timestamp(self.dan_first_frame) if self.dan_first_frame is not None else None,
            "sanbo_first_ts": self._get_timestamp(self.sanbo_first_frame) if self.sanbo_first_frame is not None else None,
            "hand_episodes": hand_summary,
            "start_kibble": peak_kibble,
            "end_kibble": end_kibble,
            "dan_kibble_eaten": dan_kibble_eaten,
            "dan_bowl_seconds": dan_bowl_seconds,
            "sanbo_kibble_eaten": sanbo_kibble_eaten,
            "sanbo_bowl_seconds": sanbo_bowl_seconds,
        }


# ───────────────────────────────────────────────────────────────
# Summary formatting helpers
# ───────────────────────────────────────────────────────────────

def format_duration(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s:02d}s"


def _parse_timestamp(ts):
    """Parse a full timestamp; return None for partial OCR garbage."""
    from datetime import datetime
    if not ts or ts in ("??", "", "unknown"):
        return None
    if not isinstance(ts, str):
        return None
    ts = ts.strip()
    # Try all common formats
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y%m%d_%H%M%S"):
        try:
            return datetime.strptime(ts, fmt)
        except ValueError:
            continue
    # Fallback to ISO
    try:
        return datetime.fromisoformat(ts.replace(" ", "T"))
    except:
        return None


def _check_cross_camera_interaction(current_date, current_camera, current_start_ts):
    """Check if the other cat was at their bowl during this feeding."""
    if 'DRIVE_CSV_DATA' not in globals() or not DRIVE_CSV_DATA:
        return ""
    
    from datetime import datetime
    now_dt = _parse_timestamp(current_start_ts)
    if not now_dt:
        return ""

    # current_camera's cat
    primary_cat = 'Sanbo' if current_camera == 'LOGITECH' else 'Dan'

    for row in DRIVE_CSV_DATA:
        if row.get('date') == current_date and row.get('camera') != current_camera:
            other_camera = row.get('camera')
            other_cat = "Dan" if other_camera == "TAPO" else "Sanbo"
            other_arrival = row.get('dan_first_arrival') if other_camera == "TAPO" else row.get('sanbo_first_arrival')
            
            if other_arrival:
                try:
                    other_dt = datetime.strptime(f"{current_date} {other_arrival}", "%Y-%m-%d %H:%M:%S")
                    if abs((now_dt - other_dt).total_seconds()) < 600:
                         return f"Note: {other_cat} was at his bowl while {primary_cat} was here."
                except:
                    pass
    return ""


def _fmt_time(ts, start_date=None, unknown="unknown", with_seconds=True):
    """Return HH:MM for valid same-day timestamps; never expose raw OCR text."""
    parsed = _parse_timestamp(ts)
    if parsed is None:
        return unknown
    if start_date and parsed.strftime("%Y-%m-%d") != start_date:
        return parsed.strftime("%m-%d %H:%M" + (":%S" if with_seconds else ""))
    return parsed.strftime("%H:%M:%S" if with_seconds else "%H:%M")


def _fmt_count(value):
    if value is None or value == "??":
        return "?"
    return str(int(round(value)))


def _fmt_hand_feeding(hand_eps, start_date):
    if not hand_eps:
        return "none"
    total = sum(ep.get("kibble_added", 0) for ep in hand_eps)
    if len(hand_eps) == 1:
        t = _fmt_time(hand_eps[0].get("timestamp"), start_date)
        return f"1 session at {t}, ~{total} kibble"
    return f"{len(hand_eps)} sessions, ~{total} kibble"


def _schedule_heartbeat_text():
    import os
    cron = os.environ.get("GHA_SCHEDULE_CRON", "")
    started = os.environ.get("GHA_JOB_STARTED_AT_UTC", "")
    delay = os.environ.get("GHA_SCHEDULE_DELAY_MIN", "")
    if not cron and not started and not delay:
        return ""
    if cron:
        label = f"cron {cron}"
    else:
        label = "manual run"
    started_time = started[11:19] if len(started) >= 19 else "unknown"
    delay_text = f"+{delay}m" if delay else "n/a"
    return f"Schedule: {label}; start {started_time} UTC; delay {delay_text}"


def _kibble_bar(count, total, width=8):
    if total <= 0:
        return "\u2591" * width
    filled = min(width, round(count / total * width))
    return "\u2588" * filled + "\u2591" * (width - filled)


def format_feeding_summary(summary, video_name):
    """Format feeding summary for Telegram: 5-second mobile scan."""
    cam_label = f"[{CAMERA_NAME}] " if "CAMERA_NAME" in globals() else ""
    start_dt = _parse_timestamp(summary.get("start_ts"))
    if not start_dt and globals().get('DATE_OVERRIDE'):
        # Force a date from override if possible
        d_str = str(DATE_OVERRIDE)
        if len(d_str) >= 8:
             try:
                 from datetime import datetime
                 start_dt = datetime.strptime(d_str[:8], "%Y%m%d")
             except: pass
    start_date = start_dt.strftime("%Y-%m-%d") if start_dt else None
    start_time = _fmt_time(summary.get("start_ts"), start_date)
    end_time = _fmt_time(summary.get("end_ts"), start_date)
    duration = format_duration(summary["duration_sec"])

    # Determine primary/other cat based on camera
    if globals().get('CAMERA_NAME') == 'LOGITECH':
        primary_cat, other_cat = 'Sanbo', 'Dan'
    else:
        primary_cat, other_cat = 'Dan', 'Sanbo'

    dan_ate = int(round(summary.get("dan_kibble_eaten", 0)))
    sanbo_ate = int(round(summary.get("sanbo_kibble_eaten", 0)))
    total_eaten = dan_ate + sanbo_ate
    dan_pct = round(dan_ate / total_eaten * 100) if total_eaten else 0
    sanbo_pct = 100 - dan_pct if total_eaten else 0
    
    primary_ate = sanbo_ate if primary_cat == 'Sanbo' else dan_ate
    other_ate = dan_ate if primary_cat == 'Sanbo' else sanbo_ate
    primary_pct = sanbo_pct if primary_cat == 'Sanbo' else dan_pct
    other_pct = dan_pct if primary_cat == 'Sanbo' else sanbo_pct

    start_kibble = _fmt_count(summary.get("start_kibble"))
    hand_eps = summary.get("hand_episodes", [])

    dan_bowl = format_duration(summary.get("dan_bowl_seconds", 0))
    sanbo_bowl = format_duration(summary.get("sanbo_bowl_seconds", 0))
    dan_seen = _fmt_time(summary.get("dan_first_ts"), start_date)
    sanbo_seen = _fmt_time(summary.get("sanbo_first_ts"), start_date)
    
    primary_bowl = sanbo_bowl if primary_cat == 'Sanbo' else dan_bowl
    other_bowl = dan_bowl if primary_cat == 'Sanbo' else sanbo_bowl
    primary_seen = sanbo_seen if primary_cat == 'Sanbo' else dan_seen
    other_seen = dan_seen if primary_cat == 'Sanbo' else sanbo_seen

    # Success/Failure logic
    if other_ate > 2: # threshold for "stole"
        first_line = f"{cam_label}😿 {other_cat} stole {primary_cat}'s food!"
    elif primary_ate > 0:
        first_line = f"{cam_label}😸 {primary_cat} finished breakfast"
    else:
        first_line = f"{cam_label}🍽️? Feeding machine not working?"

    lines = [first_line]
    lines.append(f"{start_date or 'unknown date'}")
    lines.append(f"{start_time}-{end_time} ({duration})")
    lines.append(f"Start: ~{start_kibble} kibble")

    # Prioritize primary cat details
    p_bar = _kibble_bar(primary_ate, total_eaten)
    lines.append(f"{primary_cat:<6} {p_bar} {primary_pct}% (~{primary_ate})")
    p_seen_label = f"~{primary_seen}" if primary_seen != "unknown" else primary_seen
    lines.append(f"      bowl {primary_bowl}; bowl from {p_seen_label}")
    
    o_bar = _kibble_bar(other_ate, total_eaten)
    lines.append(f"{other_cat:<6} {o_bar} {other_pct}% (~{other_ate})")
    o_seen_label = f"~{other_seen}" if other_seen != "unknown" else other_seen
    lines.append(f"      bowl {other_bowl}; bowl from {o_seen_label}")

    lines.append(f"Hand: {_fmt_hand_feeding(hand_eps, start_date)}")
    if "merged_names" in summary and summary["merged_names"]:
        count = len(summary["merged_names"])
        lines.append(f"Clips: {count} merged")

    # Cross-camera interaction
    interaction = _check_cross_camera_interaction(start_date, globals().get('CAMERA_NAME'), summary.get("start_ts"))
    if interaction:
        lines.append(interaction)

    if summary.get("flag_summary_text"):
        lines.append(summary["flag_summary_text"])

    return "\n".join(lines)


def plot_video_timeline(tracker, video_name):
    """Plot timeline chart: kibble count, cat presence, Dan_hand events."""
    n = len(tracker.kibble_counts)
    if n == 0:
        return None
    frames = np.arange(n)
    time_sec = frames / tracker.fps

    fig, axes = plt.subplots(4, 1, figsize=(16, 8), sharex=True, facecolor="#1a1a1a")
    fig.suptitle(f"Detection Timeline: {video_name}", color="white", fontsize=13, fontweight="bold")

    # Kibble count
    axes[0].fill_between(time_sec, tracker.kibble_counts, color="#FCF400", alpha=0.7)
    max_k = max(tracker.kibble_counts) if tracker.kibble_counts else 1
    axes[0].set_ylim(0, max(max_k * 1.15, 5))
    axes[0].text(time_sec[-1], max_k, f"  max {max_k}", color="#FCF400", fontsize=9, va="center")
    axes[0].set_ylabel("Kibble", color="white", fontsize=10)
    axes[0].set_facecolor("#1a1a1a")
    axes[0].tick_params(colors="white")

    # Dan at bowl
    dan_arr = np.array(tracker.dan_at_bowl, dtype=float)
    axes[1].fill_between(time_sec, dan_arr, color="#00FFCE", alpha=0.7, step="mid")
    axes[1].set_ylabel("Dan\nat bowl", color="white", fontsize=10)
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_facecolor("#1a1a1a")
    axes[1].tick_params(colors="white")

    # Sanbo at bowl
    sanbo_arr = np.array(tracker.sanbo_at_bowl, dtype=float)
    axes[2].fill_between(time_sec, sanbo_arr, color="#FF8000", alpha=0.7, step="mid")
    axes[2].set_ylabel("Sanbo\nat bowl", color="white", fontsize=10)
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].set_facecolor("#1a1a1a")
    axes[2].tick_params(colors="white")

    # Dan_hand
    hand_arr = np.array(tracker.dan_hand_present, dtype=float)
    axes[3].fill_between(time_sec, hand_arr, color="#0010FF", alpha=0.7, step="mid")
    axes[3].set_ylabel("Dan\nhand", color="white", fontsize=10)
    axes[3].set_ylim(-0.1, 1.1)
    axes[3].set_xlabel("Time (seconds)", color="white", fontsize=10)
    axes[3].set_facecolor("#1a1a1a")
    axes[3].tick_params(colors="white")

    plt.tight_layout()
    timeline_path = os.path.join(OUTPUT_DIR, f"timeline_{Path(video_name).stem}.jpg")
    plt.savefig(timeline_path, dpi=120, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print(f"✅ Timeline saved → {timeline_path}")
    return timeline_path


# ───────────────────────────────────────────────────────────────
# Telegram sender
# ───────────────────────────────────────────────────────────────

def _compress_video_for_telegram(video_path):
    """Compress video to under 50 MB using ffmpeg. Returns compressed path or None."""
    import subprocess
    compressed = video_path.replace(".mp4", "_tg.mp4")
    result = subprocess.run(
        ["ffmpeg", "-i", video_path,
         "-vcodec", "libx264", "-crf", "32", "-preset", "fast",
         "-vf", "scale=-2:480,setpts=0.25*PTS",
         "-r", "16", "-an",        # cap at 720p
         "-movflags", "+faststart",
         "-y", compressed],
        capture_output=True, text=True, timeout=300,
    )
    if result.returncode == 0 and os.path.exists(compressed):
        size_mb = os.path.getsize(compressed) / 1e6
        print(f"   Compressed: {size_mb:.0f} MB → {compressed}")
        if True: # Always return compressed
            return compressed
    return None


def send_telegram_summary(bot_token, chat_id, summary_text, snapshot_paths,
                          video_path=None, timeline_path=None):
    """Send feeding summary, timeline, snapshots, and annotated video via Telegram."""
    if not bot_token or not chat_id:
        print("⚠️  BOT_TOKEN / MY_CHAT_ID not set. Skipping Telegram notification.")
        return False

    base = f"https://api.telegram.org/bot{bot_token}"
    ok = True

    # 1. Text summary
    r = requests.post(f"{base}/sendMessage", data={
        "chat_id": chat_id,
        "text": summary_text[:4096],
    })
    if r.status_code != 200:
        print(f"❌ Telegram text error: {r.text[:200]}")
        return False
    print("✅ Telegram summary sent")

    # 2. Timeline chart
    if timeline_path and os.path.exists(timeline_path):
        try:
            with open(timeline_path, "rb") as fh:
                r = requests.post(
                    f"{base}/sendPhoto",
                    data={"chat_id": chat_id},
                    files={"photo": (Path(timeline_path).name, fh, "image/jpeg")},
                )
            if r.status_code != 200:
                print(f"⚠️  Timeline photo failed: {r.text[:100]}")
                ok = False
            else:
                print("✅ Timeline sent")
        except Exception as e:
            print(f"⚠️  Timeline error: {e}")
            ok = False

    # 3. Snapshots
    for path in snapshot_paths:
        try:
            with open(path, "rb") as fh:
                r = requests.post(
                    f"{base}/sendPhoto",
                    data={"chat_id": chat_id},
                    files={"photo": (Path(path).name, fh, "image/jpeg")},
                )
            if r.status_code != 200:
                print(f"⚠️  Photo failed ({Path(path).name}): {r.text[:100]}")
                ok = False
            else:
                print(f"✅ Snapshot sent: {Path(path).name}")
        except Exception as e:
            print(f"⚠️  Photo error ({Path(path).name}): {e}")
            ok = False

    # 4. Annotated video — compress first if over 50 MB
    if video_path and os.path.exists(video_path):
        send_path = video_path
        size_mb = os.path.getsize(video_path) / 1e6
        print(f"   Compressing {size_mb:.1f} MB video for Telegram (4x speed, 480p)...")
        compressed = _compress_video_for_telegram(video_path)
        if compressed:
            send_path = compressed
        else:
                requests.post(f"{base}/sendMessage", data={
                    "chat_id": chat_id,
                    "text": f"⚠️ Video too large even after compression ({size_mb:.0f} MB).\nDrive path: {video_path}",
                })
                print(f"⚠️  Could not compress video — Drive path sent instead")
                return ok
        try:
            with open(send_path, "rb") as fh:
                r = requests.post(
                    f"{base}/sendVideo",
                    data={"chat_id": chat_id},
                    files={"video": (Path(send_path).name, fh, "video/mp4")},
                    timeout=180,
                )
            if r.status_code != 200:
                print(f"⚠️  Video send failed: {r.text[:100]}")
                ok = False
            else:
                print(f"✅ Video sent: {Path(video_path).name}")
        except Exception as e:
            print(f"⚠️  Video error: {e}")
            ok = False

    return ok


print("✅ FeedingTracker, formatters, and Telegram sender defined.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Scan SOURCE_DIR + Process Images
# ───────────────────────────────────────────────────────────────

all_files = sorted(Path(SOURCE_DIR).iterdir())
image_paths = [f for f in all_files if classify_file(f) == "image"]
if not RUNNING_IN_CI:
    video_paths = [f for f in all_files if classify_file(f) == "video"]

print(f"✅ Found {len(image_paths)} image(s) and {len(video_paths)} video(s) in SOURCE_DIR")

if not image_paths:
    print(f"   No images found in: {SOURCE_DIR}")
else:
    print(f"\n{'='*65}")
    print(f"  Processing {len(image_paths)} image(s)...")
    print(f"{'='*65}")

    all_stats = {}
    color_count = 0
    ir_count = 0

    for img_path in image_paths:
        img_name = img_path.name
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"\u26a0\ufe0f  Could not read: {img_name} \u2014 skipping")
            continue

        image_type = detect_image_type(img_bgr)
        if image_type == "color":
            color_count += 1
        else:
            ir_count += 1

        inference_img = prepare_for_inference(img_bgr, image_type)
        tmp_path = "/tmp/_infer_tmp.jpg"
        cv2.imwrite(tmp_path, inference_img)

        results = model.predict(
            source=tmp_path,
            imgsz=IMGSZ,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            rect=True,
            verbose=False,
        )

        detections = parse_results(results)
        confs_by_class = {}
        for det in detections:
            confs_by_class.setdefault(det["class_name"], []).append(det["conf"])

        stats_by_class = {name: compute_stats(confs) for name, confs in confs_by_class.items()}
        for name, confs in confs_by_class.items():
            all_stats.setdefault(name, []).extend(confs)

        img_plain = draw_boxes(img_bgr, detections, show_label=False)
        img_labeled = draw_boxes(img_bgr, detections, show_label=True)

        stem = Path(img_name).stem
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"{stem}_boxes_only.jpg"), img_plain)
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"{stem}_with_conf.jpg"), img_labeled)

        show_dual_output(img_name, image_type, img_plain, img_labeled, stats_by_class)

    # Final summary table for all images
    if all_stats:
        render_final_table(
            total_images=len(image_paths),
            color_count=color_count,
            ir_count=ir_count,
            all_stats=all_stats,
        )

    print(f"\n✅  Image outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Phase 1: YOLO Inference & Cache  (run once per video — slow)
# ───────────────────────────────────────────────────────────────
# Runs YOLO on every Nth frame, saves raw detections + timestamps
# + compressed JPEG frames to a pickle file.
# Also writes the annotated video.
#
# Cache includes JPEG-compressed frames so Phase 2 never needs
# to re-open the video file (fast snapshot generation).
#
# If cache exists and FORCE_RERUN is False, skips inference.

if not RUNNING_IN_CI:
    all_files = sorted(Path(SOURCE_DIR).iterdir())
    video_paths = [f for f in all_files if classify_file(f) == "video"]
print(f"✅ Found {len(video_paths)} video(s) to process")

if not video_paths:
    print(f"   No videos found in: {SOURCE_DIR}")
else:
    for vid_path in video_paths:
        vid_name = vid_path.name
        vid_stem = vid_path.stem
        cache_path = os.path.join(OUTPUT_DIR, f"{vid_stem}_detections.pkl")

        # Check cache
        if os.path.exists(cache_path) and not FORCE_RERUN:
            print(f"\n✅ Cache exists for {vid_name} — skipping inference")
            print(f"   Cache: {cache_path}")
            continue

        print(f"\n{'='*65}")
        print(f"  Phase 1: Detecting — {vid_name}")
        print(f"{'='*65}")

        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            print(f"⚠️  Could not open: {vid_name} — skipping")
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        effective_fps = fps / FRAME_SKIP
        frames_to_process = total_frames // FRAME_SKIP
        print(f"   {width}x{height} @ {fps:.1f} fps, {total_frames} frames")
        print(f"   Frame skip: {FRAME_SKIP} → ~{frames_to_process} frames at ~{effective_fps:.1f} eFPS")

        # Output video writer
        out_name = f"{vid_stem}_annotated.mp4"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_path, fourcc, effective_fps, (width, height))

        # Collect detections for cache
        cached_frames = []
        last_ts = ""

        # Empty bowl early exit
        empty_bowl_threshold = int(EMPTY_BOWL_EXIT_SECONDS * effective_fps)
        empty_bowl_counter = 0

        for frame_idx in tqdm(range(total_frames), desc=vid_name, unit="frame"):
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % FRAME_SKIP != 0:
                continue

            # OCR timestamp
            processed_idx = len(cached_frames)
            if processed_idx % OCR_EVERY_N_FRAMES == 0:
                try:
                    ts_candidate = extract_timestamp(frame, filename=vid_name, frame_idx=frame_idx, fps=fps)
                    if ts_candidate:
                        last_ts = ts_candidate
                except Exception:
                    pass

            # Prepare + YOLO inference
            image_type = detect_image_type(frame)
            inference_img = prepare_for_inference(frame, image_type)
            results = model.predict(
                source=inference_img,
                imgsz=IMGSZ,
                conf=CONF_THRESHOLD,
                iou=IOU_THRESHOLD,
                verbose=False,
            )
            detections = parse_results(results)

            # Write annotated frame
            annotated = draw_boxes(frame, detections, show_label=False)
            writer.write(annotated)

            # Compress frame as JPEG for cache (~50KB vs ~9MB raw)
            _, jpeg_buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
            jpeg_bytes = jpeg_buf.tobytes()

            # Save to cache
            cached_frames.append({
                "frame_idx": frame_idx,
                "detections": detections,
                "timestamp": last_ts,
                "jpeg": jpeg_bytes,
            })

            # Empty bowl early exit check
            has_cat_at_bowl = False
            kibble_count = sum(1 for d in detections if d["class_name"] == "Kibble")
            bowl_boxes = [d for d in detections if d["class_name"] == "Bowl"]
            if bowl_boxes:
                bowl = max(bowl_boxes, key=lambda d: (d["x2"]-d["x1"])*(d["y2"]-d["y1"]))
                for d in detections:
                    if d["class_name"] in ("Dan", "Sanbo") and bbox_iou(d, bowl) > OVERLAP_IOU_THRESHOLD:
                        has_cat_at_bowl = True
                        break

            if not has_cat_at_bowl and kibble_count == 0:
                empty_bowl_counter += 1
                if empty_bowl_counter >= empty_bowl_threshold:
                    print(f"\n⏹ Empty bowl for {EMPTY_BOWL_EXIT_SECONDS}s at {last_ts or '??'} — stopping early")
                    break
            else:
                empty_bowl_counter = 0

        cap.release()
        writer.release()
        print(f"\n✅ Annotated video saved → {out_path}")

        # Save cache
        cache_data = {
            "video_name": vid_name,
            "fps": fps,
            "width": width,
            "height": height,
            "frame_skip": FRAME_SKIP,
            "effective_fps": effective_fps,
            "frames": cached_frames,
        }
        with open(cache_path, "wb") as f:
            pickle.dump(cache_data, f)

        cache_mb = os.path.getsize(cache_path) / (1024 * 1024)
        print(f"✅ Detection cache saved → {cache_path}")
        print(f"   {len(cached_frames)} frames, {cache_mb:.1f} MB (with JPEG frames)")

    print(f"\n✅  Phase 1 complete for all videos.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Phase 2: Analytics & Event Detection  (fast — re-run freely!)
# ───────────────────────────────────────────────────────────────
# Loads cached detections + JPEG frames (no video file needed).
# Runs FeedingTracker with current tunable parameters from Config.
#
# 💡 ITERATION WORKFLOW:
#    1. Change thresholds in the Config cell above
#    2. Re-run THIS cell only (~2 seconds)
#    3. Check results — repeat until satisfied
#    4. Then run Cell 11 (Output) to save & send
#
# Tunable params that affect this cell:
#   - SANBO_MIN_CONSECUTIVE_FRAMES  (Sanbo false alarm filter)
#   - DAN_HAND_CONF_THRESHOLD       (hand feeding sensitivity)
#   - DAN_HAND_GAP_SECONDS          (episode merging)
#   - DAN_HAND_MIN_SECONDS          (min episode duration)
#   - OVERLAP_IOU_THRESHOLD         (cat-at-bowl sensitivity)
#   - EATING_KIBBLE_DROP            (min kibble drop to confirm eating)

if not RUNNING_IN_CI:
    all_files = sorted(Path(SOURCE_DIR).iterdir())
    video_paths = [f for f in all_files if classify_file(f) == "video"]
print(f"✅ Found {len(video_paths)} video(s)")

# This list is consumed by Cell 11 (Output)
video_results = []

total_hand_episodes = 0
food_finished_seen = False
for vid_path in video_paths:
    vid_name = vid_path.name
    vid_stem = vid_path.stem
    cache_path = os.path.join(OUTPUT_DIR, f"{vid_stem}_detections.pkl")

    if not os.path.exists(cache_path):
        print(f"\n⚠️  No cache for {vid_name} — run Phase 1 first")
        continue

    # Load cache
    with open(cache_path, "rb") as f:
        cache = pickle.load(f)

    effective_fps = cache["effective_fps"]
    n_cached = len(cache["frames"])
    has_jpegs = "jpeg" in cache["frames"][0] if cache["frames"] else False
    print(f"\n{'='*65}")
    print(f"  Phase 2: Analyzing — {vid_name}")
    print(f"  ({n_cached} cached frames, eFPS={effective_fps:.1f}, JPEG={'✅' if has_jpegs else '❌'})")
    print(f"{'='*65}")

    # Create tracker with current tunable parameters
    tracker = FeedingTracker(
        fps=effective_fps,
        sanbo_min_frames=SANBO_MIN_CONSECUTIVE_FRAMES,
        episode_offset=total_hand_episodes,
    )

    # Feed cached detections to tracker
    for i, frame_data in enumerate(cache["frames"]):
        detections = frame_data["detections"]
        timestamp = frame_data["timestamp"]

        # Decode frame from JPEG cache (no video seek needed)
        if has_jpegs and "jpeg" in frame_data:
            jpeg_arr = np.frombuffer(frame_data["jpeg"], dtype=np.uint8)
            frame = cv2.imdecode(jpeg_arr, cv2.IMREAD_COLOR)
        else:
            # Fallback: blank frame (old cache without JPEGs)
            frame = np.zeros((cache["height"], cache["width"], 3), dtype=np.uint8)

        tracker.process_frame(i, detections, timestamp, frame)

    # --- Summary ---
    summary = tracker.summarize()
    total_hand_episodes += len(summary.get("hand_episodes", []))
    if RUNNING_IN_CI and 'merged_sources' in globals() and vid_name in merged_sources:
        summary['merged_names'] = merged_sources[vid_name]
    total_eaten = int(summary.get("dan_kibble_eaten", 0)) + int(summary.get("sanbo_kibble_eaten", 0))
    start_kibble = summary.get("start_kibble")
    end_kibble = summary.get("end_kibble")
    empty_threshold = int(globals().get("FOOD_FINISHED_KIBBLE_MAX", 1))
    if food_finished_seen and total_eaten == 0 and (start_kibble is None or start_kibble <= empty_threshold):
        print(f"\nSkipping {vid_name}: food was already finished in an earlier event.")
        continue

    summary_text = format_feeding_summary(summary, vid_name)
    print(f"\n{summary_text}")

    if total_eaten > 0 and end_kibble is not None and end_kibble <= empty_threshold:
        food_finished_seen = True

    # Store results for Cell 11
    video_results.append({
        "vid_name": vid_name,
        "vid_stem": vid_stem,
        "tracker": tracker,
        "summary": summary,
        "summary_text": summary_text,
    })

print(f"\n✅  Phase 2 complete.")
print(f"   💡 Adjust params in Config and re-run THIS cell to iterate!")
print(f"   📤 Run the next cell (Output) to save files & send to Telegram.")

In [ ]:
# Phase 2.5 — Auto-flag suspicious detections
from flagging import flag_detections
import pickle

FLAG_CONF_THRESHOLD = 0.40
FLAG_BLIP_MAX_FRAMES = 2
FLAG_BLIP_GAP_FRAMES = 5
FLAG_IOU_CONFLICT = 0.50
FLAG_KIBBLE_JUMP = 15
FLAG_DEDUP_WINDOW = 3

all_flagged = {}

for vr in video_results:
    vid_stem = vr['vid_stem']
    cache_path = os.path.join(OUTPUT_DIR, f"{vid_stem}_detections.pkl")
    if not os.path.exists(cache_path):
        print(f"  [skip] No cache for {vid_stem}")
        continue

    with open(cache_path, 'rb') as f:
        cache = pickle.load(f)

    flagged = flag_detections(
        cache['frames'],
        conf_threshold=FLAG_CONF_THRESHOLD,
        blip_max_frames=FLAG_BLIP_MAX_FRAMES,
        blip_gap_frames=FLAG_BLIP_GAP_FRAMES,
        iou_conflict=FLAG_IOU_CONFLICT,
        kibble_jump=FLAG_KIBBLE_JUMP,
        dedup_window=FLAG_DEDUP_WINDOW,
    )
    all_flagged[vid_stem] = flagged
    print(f"  {vid_stem}: {len(flagged)} frames flagged")

total_flagged = sum(len(v) for v in all_flagged.values())
print(f"\nTotal flagged: {total_flagged} frames")

In [ ]:
# Phase 2.6 — Upload flagged frames to Roboflow
from roboflow_upload import upload_flagged_frames, format_telegram_flag_summary, UploadResult

total_flagged = sum(len(v) for v in all_flagged.values()) if 'all_flagged' in dir() else 0
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
ROBOFLOW_WORKSPACE = os.environ.get('ROBOFLOW_WORKSPACE', '')
ROBOFLOW_PROJECT = 'ir-kibble'

combined_result = UploadResult()

if not ROBOFLOW_API_KEY:
    print("ROBOFLOW_API_KEY not set — skipping Roboflow upload")
    flag_summary_text = "Roboflow upload skipped (no API key)"
elif total_flagged == 0:
    print("No frames flagged — nothing to upload")
    flag_summary_text = format_telegram_flag_summary(combined_result, total_flagged)
else:
    for vid_stem, flagged in all_flagged.items():
        if not flagged:
            continue
        print(f"  Uploading {len(flagged)} frames from {vid_stem}...")
        
        result = upload_flagged_frames(
            flagged,
            api_key=ROBOFLOW_API_KEY,
            workspace=ROBOFLOW_WORKSPACE,
            project=ROBOFLOW_PROJECT,
            video_stem=vid_stem,
            batch_name=f"flagged-logitech-{datetime.now().strftime('%Y-%m')}" if CAMERA_NAME == "LOGITECH" else None,
        )
        combined_result.uploaded += result.uploaded
        combined_result.failed += result.failed
        combined_result.skipped += result.skipped
        for tag, count in result.tag_counts.items():
            combined_result.tag_counts[tag] = combined_result.tag_counts.get(tag, 0) + count

    flag_summary_text = format_telegram_flag_summary(combined_result, total_flagged)
    print(f"\n{flag_summary_text}")

if 'video_results' in globals():
    for vr in video_results:
        vr["summary"]["flag_summary_text"] = flag_summary_text
        vr["summary_text"] = format_feeding_summary(vr["summary"], vr["vid_name"])


In [ ]:
# ───────────────────────────────────────────────────────────────
# Phase 3: Output & Telegram  (save files, charts, send)
# ───────────────────────────────────────────────────────────────
# Takes results from Phase 2, saves text summaries, snapshots,
# timeline charts, and sends everything to Telegram.
#
# 📌 Re-run this cell after Phase 2 to update outputs.
# 📌 Also safe to re-run independently (e.g. if Telegram failed).

if 'video_results' not in dir() or not video_results:
    print("⚠️  No video_results — run Phase 2 (Analytics) first.")
else:
    video_summaries = []

    for vr in video_results:
        vid_name = vr["vid_name"]
        vid_stem = vr["vid_stem"]
        tracker = vr["tracker"]
        summary_text = vr["summary_text"]

        print(f"\n{'='*65}")
        print(f"  Phase 3: Output — {vid_name}")
        print(f"{'='*65}")

        # --- Save text summary ---
        txt_path = os.path.join(OUTPUT_DIR, f"{vid_stem}_summary.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(summary_text)
        print(f"✅ Summary saved → {txt_path}")

        # --- Save snapshots ---
        snapshot_paths = []
        for event_name, snap_img in tracker.snapshots.items():
            snap_path = os.path.join(OUTPUT_DIR, f"{vid_stem}_{event_name}.jpg")
            cv2.imwrite(snap_path, snap_img)
            snapshot_paths.append(snap_path)
            print(f"✅ Snapshot → {snap_path}")

        # --- Timeline chart ---
        timeline_path = plot_video_timeline(tracker, vid_name)

        # --- Send to Telegram ---
        out_path = os.path.join(OUTPUT_DIR, f"{vid_stem}_annotated.mp4")
        print(f"\n📤 Sending {vid_name} to Telegram...")
        send_telegram_summary(
            BOT_TOKEN,
            MY_CHAT_ID,
            summary_text,
            snapshot_paths,
            video_path=out_path if os.path.exists(out_path) else None,
            timeline_path=timeline_path,
        )

        video_summaries.append({
            "video_name": vid_name,
            "summary_text": summary_text,
            "snapshot_paths": snapshot_paths,
            "video_path": out_path,
            "timeline_path": timeline_path,
        })

        # --- Display video inline ---
        if os.path.exists(out_path):
            try:
                display(Video(out_path, embed=True, width=800))
            except Exception:
                print("   (Video display not supported inline)")

    print(f"\n✅  Phase 3 complete. All outputs saved & sent.")

In [ ]:
# -- Append to feeding_log.csv on Drive --------------------------------
# NOTE: Runs AFTER Phase 3. video_results must be populated.
import csv, os
from schedule_log import current_schedule_fields, fetch_schedule_lookup
from datetime import date, datetime
from pathlib import Path as _Path

_LOG_FILE    = _Path('feeding_log.csv')
_WEIGHT_FILE = _Path('weight_log.csv')
_FIELDS = [
    'date', 'camera', 'dan_kibble', 'sanbo_kibble', 'hand_feeding', 'compensation',
    'video_count', 'dan_first_arrival', 'sanbo_first_arrival',
    'schedule_time', 'start_time',
    'flagged_frames', 'roboflow_uploaded', 'roboflow_skipped', 'roboflow_failed', 'flag_top_tags',
    'dan_weight', 'sanbo_weight',
]

_skip_csv = not ('video_results' in globals() and video_results)
if _skip_csv:
    print('No video_results -- skipping CSV log (no videos processed today)')

# -- Aggregate across ALL feeding events --------------------------------
# Use override if present, otherwise today (formatted as YYYY-MM-DD for the CSV)
if 'DATE_OVERRIDE' in globals() and DATE_OVERRIDE:
    _today_str = str(DATE_OVERRIDE); _today = f"{_today_str[:4]}-{_today_str[4:6]}-{_today_str[6:8]}"
else:
    _today = str(date.today())
_dan_kibble        = 0
_sanbo_kibble      = 0
_hand_feeding      = 0
_compensation      = 0
_vcount            = len(video_results) if not _skip_csv else 0
_dan_first_arrival   = ''
_sanbo_first_arrival = ''
_flagged_frames = total_flagged if 'total_flagged' in globals() else ''
_roboflow_uploaded = combined_result.uploaded if 'combined_result' in globals() else ''
_roboflow_skipped = combined_result.skipped if 'combined_result' in globals() else ''
_roboflow_failed = combined_result.failed if 'combined_result' in globals() else ''
_schedule_fields = current_schedule_fields(os.environ, _today)
_flag_top_tags = ''
if 'combined_result' in globals() and getattr(combined_result, 'tag_counts', None):
    _flag_top_tags = '; '.join(
        f'{tag}:{count}' for tag, count in sorted(
            combined_result.tag_counts.items(), key=lambda x: x[1], reverse=True
        )[:5]
    )

if not _skip_csv:
    for _vr in video_results:
        _s = _vr.get('summary', {})
        _dan_kibble   += int(_s.get('dan_kibble_eaten', 0))
        _sanbo_kibble += int(_s.get('sanbo_kibble_eaten', 0))
        _hand_feeding += sum(e.get('kibble_added', 0) for e in _s.get('hand_episodes', []))
        _compensation += int(_s.get('sanbo_kibble_eaten', 0))
        if not _dan_first_arrival and _s.get('dan_first_ts'):
            _t = _s['dan_first_ts']
            _dan_first_arrival = _t.split()[-1] if ' ' in _t else _t
        if not _sanbo_first_arrival and _s.get('sanbo_first_ts'):
            _t = _s['sanbo_first_ts']
            _sanbo_first_arrival = _t.split()[-1] if ' ' in _t else _t

_row = {
    'date':               _today,
    'camera':             CAMERA_NAME,
    'dan_kibble':         _dan_kibble,
    'sanbo_kibble':       _sanbo_kibble,
    'hand_feeding':       _hand_feeding,
    'compensation':       _compensation,
    'video_count':        _vcount,
    'dan_first_arrival':  _dan_first_arrival,
    'sanbo_first_arrival':_sanbo_first_arrival,
    'schedule_time':      _schedule_fields.get('schedule_time', ''),
    'start_time':         _schedule_fields.get('start_time', ''),
    'flagged_frames':     _flagged_frames,
    'roboflow_uploaded':  _roboflow_uploaded,
    'roboflow_skipped':   _roboflow_skipped,
    'roboflow_failed':    _roboflow_failed,
    'flag_top_tags':      _flag_top_tags,
    'dan_weight':         '',
    'sanbo_weight':       '',
}

if RUNNING_IN_CI and not _skip_csv:
    from google.oauth2 import service_account as _sa
    from googleapiclient.discovery import build as _build
    from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
    import json as _json2

    _key2   = _json2.loads(os.environ['GDRIVE_SERVICE_ACCOUNT_KEY'])
    _creds2 = _sa.Credentials.from_service_account_info(
        _key2, scopes=['https://www.googleapis.com/auth/drive']
    )
    _drive2 = _build('drive', 'v3', credentials=_creds2)
    _out_id = os.environ['GDRIVE_OUTPUT_FOLDER_ID']
    _up_id  = os.environ['GDRIVE_UPLOAD_FOLDER_ID']

    # -- Download existing feeding_log.csv ----------------------------
    _existing_csv = _drive2.files().list(
        q=f"'{_out_id}' in parents and name='feeding_log.csv' and trashed=false",
        fields='files(id)'
    ).execute().get('files', [])

    _existing_rows = []
    if _existing_csv:
        _req = _drive2.files().get_media(fileId=_existing_csv[0]['id'])
        with open(_LOG_FILE, 'wb') as _fh:
            _dl = MediaIoBaseDownload(_fh, _req)
            _done = False
            while not _done:
                _, _done = _dl.next_chunk()
        print(f'Downloaded existing feeding_log.csv ({_LOG_FILE.stat().st_size} bytes)')
        with open(_LOG_FILE, newline='', encoding='utf-8') as _f:
            _existing_rows = list(csv.DictReader(_f))
            DRIVE_CSV_DATA = _existing_rows  # Update global cache

    # -- Download weight_log.csv from upload folder -------------------
    _weight_rows = []
    _w_search = _drive2.files().list(
        q=f"'{_up_id}' in parents and name='weight_log.csv' and trashed=false",
        fields='files(id)'
    ).execute().get('files', [])
    if _w_search:
        _wreq = _drive2.files().get_media(fileId=_w_search[0]['id'])
        with open(_WEIGHT_FILE, 'wb') as _wfh:
            _wdl = MediaIoBaseDownload(_wfh, _wreq)
            _wdone = False
            while not _wdone:
                _, _wdone = _wdl.next_chunk()
        with open(_WEIGHT_FILE, newline='', encoding='utf-8') as _wf:
            _weight_rows = list(csv.DictReader(_wf))
        print(f'Loaded {len(_weight_rows)} weight entries from weight_log.csv')
    else:
        print('weight_log.csv not found on Drive -- weight columns will be empty')

    # -- Build weight lookup by date ---------------------------------
    _w_by_date = {}
    for _wr in _weight_rows:
        _d = _wr.get('date', '').strip()
        _c = _wr.get('cat', '').strip().lower()
        _w = _wr.get('weight_kg', '').strip()
        if _d and _c and _w:
            _w_by_date.setdefault(_d, {})[_c] = _w

    # Merge today's weight
    if _today in _w_by_date:
        _row['dan_weight']   = _w_by_date[_today].get('dan', '')
        _row['sanbo_weight'] = _w_by_date[_today].get('sanbo', '')

    # -- Backfill GitHub Actions schedule/start times ----------------
    _schedule_lookup = fetch_schedule_lookup(
        os.environ.get('GITHUB_TOKEN', ''),
        os.environ.get('GITHUB_REPOSITORY', ''),
        'morning-report.yml',
        os.environ.get('GHA_SCHEDULE_CRON', '0 3 * * *'),
    )
    if _schedule_lookup:
        print(f'Loaded {len(_schedule_lookup)} GitHub scheduled run timestamps')
    for _r2 in _existing_rows:
        _d2 = _r2.get('date', '')
        if _d2 in _schedule_lookup:
            if not _r2.get('schedule_time'):
                _r2['schedule_time'] = _schedule_lookup[_d2].get('schedule_time', '')
            if not _r2.get('start_time'):
                _r2['start_time'] = _schedule_lookup[_d2].get('start_time', '')

    # -- Dedup: remove today's row for THIS camera, append fresh one --
    _existing_rows = [r for r in _existing_rows if (r.get('date') != _today or r.get('camera') != CAMERA_NAME)]

    # Backfill weight into all historical rows
    for _r2 in _existing_rows:
        _d2 = _r2.get('date', '')
        if _d2 in _w_by_date:
            if not _r2.get('dan_weight'):
                _r2['dan_weight']   = _w_by_date[_d2].get('dan', '')
            if not _r2.get('sanbo_weight'):
                _r2['sanbo_weight'] = _w_by_date[_d2].get('sanbo', '')

    _all_rows = _existing_rows + [_row]

    # -- Write updated CSV ------------------------------------------
    with open(_LOG_FILE, 'w', newline='', encoding='utf-8') as _f:
        _writer = csv.DictWriter(_f, fieldnames=_FIELDS, extrasaction='ignore')
        _writer.writeheader()
        for _r2 in _all_rows:
            for _col in _FIELDS:
                _r2.setdefault(_col, '')
            _writer.writerow(_r2)
    print(f'Logged: {_row}')

    # -- Upload feeding_log.csv ------------------------------------
    _media_csv = MediaFileUpload(str(_LOG_FILE), mimetype='text/csv')
    if _existing_csv:
        _drive2.files().update(
            fileId=_existing_csv[0]['id'], media_body=_media_csv
        ).execute()
        print('Updated feeding_log.csv on Drive')
    else:
        try:
            _drive2.files().create(
                body={'name': 'feeding_log.csv', 'parents': [_out_id]},
                media_body=_media_csv
            ).execute()
            print('Created feeding_log.csv on Drive')
        except Exception as _csv_err:
            print(f'Could not create feeding_log.csv: {_csv_err}')

    # -- 30-day weight reminder ------------------------------------
    _valid_dates = [
        datetime.strptime(_wr['date'], '%Y-%m-%d')
        for _wr in _weight_rows if _wr.get('date', '').strip()
    ]
    if _valid_dates:
        _last_w = max(_valid_dates)
        _days_since = (datetime.today() - _last_w).days
        if _days_since >= 30:
            _tg_token = os.environ.get('TelegramBotToken', '')
            _tg_chat  = os.environ.get('TelegramChatId', '')
            if _tg_token and _tg_chat:
                import requests as _req_mod
                _req_mod.post(
                    f'https://api.telegram.org/bot{_tg_token}/sendMessage',
                    json={'chat_id': _tg_chat, 'text':
                        f'Weight Reminder\n'
                        f'Last entry was {_days_since} days ago ({_last_w.strftime("%Y-%m-%d")}).\n'
                        f'Use /weight to log Dan and Sanbo current weight!'},
                    timeout=10
                )
                print(f'Weight reminder sent ({_days_since} days since last entry)')
    else:
        print('No weight entries yet -- skipping reminder check')

elif not _skip_csv:
    _write_header = not _LOG_FILE.exists()
    with open(_LOG_FILE, 'a', newline='', encoding='utf-8') as _f:
        _writer = csv.DictWriter(_f, fieldnames=_FIELDS)
        if _write_header:
            _writer.writeheader()
        _writer.writerow(_row)
    print(f'Logged locally: {_row}')


In [ ]:
if RUNNING_IN_CI:
    print('ℹ️ CI mode: skipping retry cell (Phase 3 already sent)')
else:
    # ───────────────────────────────────────────────────────────────
    # Telegram Retry  (optional — run this cell to re-send if cell 14 failed)
    # ───────────────────────────────────────────────────────────────
    # Cell 14 already sends each video to Telegram as it finishes.
    # Run this cell only if you need to re-send (e.g. network failure).
    
    if 'video_summaries' not in dir() or not video_summaries:
        print("No video_summaries in memory. Run cell 14 first.")
    else:
        print(f"Re-sending {len(video_summaries)} summary(ies) to Telegram...\n")
        for vs in video_summaries:
            print(f"── {vs['video_name']} ──")
            send_telegram_summary(
                BOT_TOKEN,
                MY_CHAT_ID,
                vs["summary_text"],
                vs["snapshot_paths"],
                video_path=vs.get("video_path"),
                timeline_path=vs.get("timeline_path"),
            )
            print()

## Results

All annotated outputs are saved to the output directory on Google Drive.

- **Images**: `{filename}_boxes_only.jpg` + `{filename}_with_conf.jpg`
- **Videos**: `{filename}_annotated.mp4` (boxes only — no labels/percentages)
- **Snapshots**: `{filename}_sanbo_arrival.jpg`, `{filename}_dan_hand_ep0.jpg`, `{filename}_kibble_dispensed_ep0.jpg`, etc.
- **Text summary**: `{filename}_summary.txt` (saved to Google Drive)
- **Charts**: `timeline_{filename}.jpg`, `SUMMARY_TABLE.jpg`

The feeding summary is printed above, saved as `.txt`, and sent to Telegram (if secrets are configured in Infisical).

### Summary format
```
🐱 Fair Feeder Summary — video.mp4
   2024-01-15 18:30:00 → 19:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📦 Feeding (Dan_hand): 2 attempt(s)
   18:32:15 — ~5 kibble added
   18:45:30 — ~3 kibble added

🐈 Sanbo arrived at 18:35:42
🐈‍⬛ Dan present from 18:30:05

🍽️ Eating estimate:
   Dan:   ~12 kibble (at bowl 5m 30s)
   Sanbo: ~8 kibble  (at bowl 3m 15s)

✋ Dan's hand attempts: 2
```